In [19]:
import re
import requests
import pandas as pd

In [20]:
IDENTIFY_UNIPROT = re.compile(r"^(?:[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2}|[OPQ][0-9][A-Z0-9]{3}[0-9])(?:-\d+)?$")
IDENTIFY_ENSEMBL = re.compile(r"^ENS[A-Z]{0,10}(?:E|FM|G|GT|P|R|T)\d{6,}(?:\.\d+)?$")

In [21]:
# fetches data from the UniProtKB REST API using accession IDs
def get_uniprot(uniprot_id: str):
    api_url = "https://rest.uniprot.org/uniprotkb/accessions"
    query_params = {"accessions": uniprot_id}
    return requests.get(api_url, params=query_params, timeout=25)

In [22]:
# extracts organism, gene, sequence and type info from UniProt response
def uniprot_parse_response(resp):
    try:
        json_data = resp.json()
        records = json_data.get("results", [])

        if not records:
            return {"err_msg": "No UniProt records found"}

        parsed_results = {}
        for item in records:
            accession = item.get("primaryAccession")
            parsed_results[accession] = {
                "organism": item.get("organism", {}).get("scientificName"),
                "geneInfo": item.get("genes"),
                "sequenceInfo": item.get("sequence"),
                "type": "protein"
            }
        return parsed_results
    except (KeyError, TypeError, ValueError) as err:
        return {"err_msg": f"UniProt parse error: {str(err)}"}

In [23]:
# queries the Ensembl REST API for a specific genomic feature ID
def get_ensembl(ensembl_id: str):
    api_url = f"https://rest.ensembl.org/lookup/id/{ensembl_id}"
    req_headers = {"Content-Type": "application/json"}
    return requests.get(api_url, headers=req_headers, timeout=25)

In [24]:
# extracts metadata fields from the Ensembl registry response
def ensembl_parse_response(resp):
    try:
        json_data = resp.json()
        feature_id = json_data.get("id")

        if not feature_id:
            return {"err_msg": "No Ensembl feature ID discovered"}

        extracted_fields = {
            "object_type": json_data.get("object_type"),
            "species": json_data.get("species"),
            "assembly_name": json_data.get("assembly_name"),
            "biotype": json_data.get("biotype"),
            "display_name": json_data.get("display_name"),
            "id": feature_id,
            "db_type": json_data.get("db_type"),
            "description": json_data.get("description"),
            "source": json_data.get("source"),
            "canonical_transcript": json_data.get("canonical_transcript")
        }
        return {feature_id: extracted_fields}
    except (KeyError, TypeError, ValueError) as err:
        return {"err_msg": f"Ensembl parse error: {str(err)}"}

In [25]:
# processes a list of biological IDs, handles API routing via regex and returns a structured pandas DF
def main(ids: list):
    dataset_rows = []

    for raw_id in ids:
        clean_id = raw_id.strip()
        if not clean_id:
            continue

        record_entry = {"input_id": clean_id}

        try:
            if IDENTIFY_ENSEMBL.match(clean_id):
                api_resp = get_ensembl(clean_id)
                api_resp.raise_for_status()
                parsed_data = ensembl_parse_response(api_resp)

            elif IDENTIFY_UNIPROT.match(clean_id):
                api_resp = get_uniprot(clean_id)
                api_resp.raise_for_status()
                parsed_data = uniprot_parse_response(api_resp)

            else:
                record_entry["error"] = "error:unknown database"
                dataset_rows.append(record_entry)
                continue

            if "err_msg" in parsed_data:
                record_entry["error"] = parsed_data["err_msg"]
            else:
                inner_data = parsed_data.get(clean_id, next(iter(parsed_data.values())))
                record_entry.update(inner_data)

        except requests.RequestException as network_err:
            record_entry["error"] = f"Network/HTTP error: {str(network_err)}"

        dataset_rows.append(record_entry)

    return pd.DataFrame(dataset_rows)

In [26]:
# test
results = main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618', 'MGP_AJ_G0023128'])

In [27]:
results

,input_id,organism,geneInfo,sequenceInfo,type,error,object_type,species,assembly_name,biotype,display_name,id,db_type,description,source,canonical_transcript
0,P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hello,NaN,NaN,NaN,NaN,error:unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRAF,ENSG00000157764,core,"B-Raf proto-oncogene, serine/threonine kinase ...",ensembl_havana,ENST00000646891.2
4,ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRCA2,ENSG00000139618,core,BRCA2 DNA repair associated [Source:HGNC Symbo...,ensembl_havana,ENST00000380152.8
5,MGP_AJ_G0023128,NaN,NaN,NaN,NaN,error:unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
